In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
import plotly.io as pio
from plotly.subplots import make_subplots
import math


In [3]:
master = pd.read_csv("../data/processed/master_climate_data.csv")
risk = pd.read_csv("../data/processed/climate_risk_index_full.csv")
ranking = pd.read_csv("../data/processed/climate_risk_index_ranking.csv")

In [4]:
COLORWAY = ["#0B3D63", "#1E88A8", "#4BACC6", "#F2A65A", "#D9534F"]
import plotly.io as pio
pio.templates["paradise"] = go.layout.Template(
    layout=dict(colorway=COLORWAY, font=dict(family="Georgia, serif"))
)
pio.templates.default = "plotly_dark+paradise"

# ## Chart 1 — Sea level anomaly trend, all 5 islands

In [17]:
sea_df = (
    master
    .dropna(subset=["sea_level_anomaly_m"])
    .sort_values(["country", "year"])
    .copy()
)
sea_df["sea_level_5yr_ma"] = (
    sea_df
    .groupby("country")["sea_level_anomaly_m"]
    .transform(lambda x: x.rolling(window=5, center=True, min_periods=3).mean())
)
countries = sea_df["country"].dropna().unique().tolist()
country_colors = {
    country: COLORWAY[i]
    for i, country in enumerate(countries)
}
fig1 = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=countries,
    shared_yaxes=True,
    shared_xaxes=True,
    horizontal_spacing=0.02,
    vertical_spacing=0.07
)

for i, country in enumerate(countries):

    row = (i // 3) + 1
    col = (i % 3) + 1

    country_df = sea_df[
        sea_df["country"] == country
    ].sort_values("year")

    country_color = country_colors[country]

    fig1.add_trace(
        go.Scatter(
            x=country_df["year"],
            y=country_df["sea_level_anomaly_m"],
            mode="markers",
            name="Annual observation",
            legendgroup="annual",
            showlegend=False,
            marker=dict(size=6,color=country_color,opacity=0.35),
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Sea-level anomaly: %{y:.2f} m"
                "<extra>" + country + "</extra>"
            )
        ),
        row=row,
        col=col
    )

    fig1.add_trace(
        go.Scatter(
            x=country_df["year"],
            y=country_df["sea_level_5yr_ma"],
            mode="lines",
            name="5-year moving average",
            legendgroup="moving_average",
            showlegend=False,
            line=dict(width=4,color=country_color),
            hovertemplate=(
                "<b>%{x}</b><br>"
                "5-year average: %{y:.2f} m"
                "<extra>" + country + "</extra>"
            )
        ),
        row=row,
        col=col
    )
    fig1.add_hline(
        y=0,
        line_width=1,
        line_dash="dash",
        opacity=0.35,
        row=row,
        col=col
    )

fig1.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        name="Annual observation",
        marker=dict(
            size=7,
            color="#AAAAAA"
        ),
        showlegend=True
    )
)

fig1.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        name="5-year moving average",
        line=dict(
            color="#AAAAAA",
            width=4
        ),
        showlegend=True
    )
)

fig1.update_xaxes(
    title_text="",
    matches=None
)

fig1.update_yaxes(
    title_text=""
)

fig1.update_layout(
    title="Sea-Level Anomalies and 5-Year Moving Averages",
    template="plotly_dark+paradise",
    hovermode="x unified",
    autosize=True,
    margin=dict(
        l=70,
        r=40,
        t=90,
        b=60
    )
)

fig1.add_annotation(
    text="Year",
    x=0.5,
    y=-0.15,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=14)
)

fig1.add_annotation(
    text="Sea-level anomaly (m)",
    x=-0.09,
    y=0.15,
    xref="paper",
    yref="paper",
    textangle=-90,
    showarrow=False,
    font=dict(
        family="Georgia, serif",
        size=15
    )
)

fig1.write_html("../assets/charts/chart1_sea_level.html", include_plotlyjs="cdn", full_html=False,config={"responsive": True}, default_width="100%", default_height="100%")

# ## Chart 2 — Temperature + rainfall small multiples

In [15]:
fig2 = px.line(
    master.dropna(subset=["temp_anomaly_c"]),
    x="year", y="temp_anomaly_c", color="country", facet_col="country", facet_col_wrap=3,
    title="Surface Temperature Anomalies"
)
fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig2.update_xaxes(title_text="",range=[1987, 2025])
fig2.update_yaxes(title_text="")
fig2.add_annotation(
    text="Year",
    x=0.5,
    y=-0.19,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(
        family="Georgia, serif",
        size=14
    )
)
fig2.add_annotation(
    text="Temperature Anomalies",
    x=-0.05,
    y=0.15,
    xref="paper",
    yref="paper",
    textangle=-90,
    showarrow=False,
    font=dict(
        family="Georgia, serif",
        size=15
    )
)
fig2.write_html("../assets/charts/chart2_temperature.html", include_plotlyjs="cdn", full_html=False)


# ## Chart 3 — Population growth

In [6]:
fig3 = px.bar(
    master.dropna(subset=["population_growth_pct"]).sort_values("year").groupby("country").tail(1),
    x="country", y="population_growth_pct",
    title="Most Recent Population Growth Rate by Island"
)
fig3.update_xaxes(title_text="")
fig3.update_yaxes(title_text="")
fig3.add_annotation(
    text="Country",
    x=0.5,
    y=-0.15,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(
        family="Georgia, serif",
        size=14
    )
)
fig3.add_annotation(
    text="Population Growth (%)",
    x=-0.05,
    y=0.5,
    xref="paper",
    yref="paper",
    textangle=-90,
    showarrow=False,
    font=dict(
        family="Georgia, serif",
        size=15
    )
)
fig3.write_html("../assets/charts/chart3_population.html", include_plotlyjs="cdn", full_html=False)


# ## Chart 4 — Tourism dependence

In [5]:
tourism_df = risk.dropna(subset=["tourism_per_capita"])
fig4 = px.bar(
    tourism_df, x="country", y="tourism_per_capita",
    title="Tourism Arrivals per Resident (0-100)"
)
fig4.add_annotation(text="Tuvalu: no tourism data available in official source",
                     showarrow=False, x=0.5, y=1.1, xref="paper", yref="paper")

fig4.update_xaxes(title_text="")
fig4.update_yaxes(title_text="")
fig4.add_annotation(
    text="Country",
    x=0.5,
    y=-0.15,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(
        family="Georgia, serif",
        size=14
    )
)
fig4.add_annotation(
    text="Tourism per Capita (0-100)",
    x=-0.05,
    y=0.5,
    xref="paper",
    yref="paper",
    textangle=-90,
    showarrow=False,
    font=dict(
        family="Georgia, serif",
        size=15
    )
)
fig4.write_html("../assets/charts/chart4_tourism.html", include_plotlyjs="cdn", full_html=False)


# ## Chart 5 — THE payoff visual: Climate Risk Index ranked

In [8]:
fig5 = px.bar(
    ranking.sort_values("climate_risk_index"),
    x="climate_risk_index", y="country", orientation="h",
    title="Future of Paradise Climate Risk Index by Island (0-100)",
    color="climate_risk_index", color_continuous_scale="Reds"
)

fig5.update_xaxes(title_text="")
fig5.update_yaxes(title_text="")
fig5.add_annotation(
    text="Custom Climate Risk Index",
    x=0.5,
    y=-0.15,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(
        family="Georgia, serif",
        size=14
    )
)
fig5.add_annotation(
    text="Country",
    x=-0.05,
    y=0.5,
    xref="paper",
    yref="paper",
    textangle=-90,
    showarrow=False,
    font=dict(
        family="Georgia, serif",
        size=15
    )
)
fig5.write_html("../assets/charts/chart5_risk_index.html", include_plotlyjs="cdn", full_html=False)

# ## Chart 6 — Explainability breakdown (stacked bar)

In [9]:
breakdown = ranking.melt(
    id_vars=["country"],
    value_vars=["contribution_physical_pct", "contribution_population_pct", "contribution_economic_pct"],
    var_name="component", value_name="pct"
)
breakdown["component"] = breakdown["component"].str.replace("contribution_", "").str.replace("_pct", "")

fig6 = px.bar(
    breakdown, x="pct", y="country", color="component", orientation="h",
    title="What Drives Each Island's Risk Score"
)

fig6.update_xaxes(title_text="")
fig6.update_yaxes(title_text="")
fig6.add_annotation(
    text="Percentage Contribution to Index",
    x=0.5,
    y=-0.15,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(
        family="Georgia, serif",
        size=14
    )
)
fig6.add_annotation(
    text="Country",
    x=-0.06,
    y=0.5,
    xref="paper",
    yref="paper",
    textangle=-90,
    showarrow=False,
    font=dict(
        family="Georgia, serif",
        size=15
    )
)

fig6.write_html("../assets/charts/chart6_breakdown.html", include_plotlyjs="cdn", full_html=False)

In [10]:
print("All 6 charts saved to assets/charts/")

All 6 charts saved to assets/charts/
